# Integration of OpenCellID Infrastructure Features

This notebook extends Dataset19 by integrating cellular-infrastructure information obtained from OpenCellID.

OpenCellID records are cleaned, converted into spatial points, and assigned to the labelled 1 km × 1 km grid cells. Six grid-level infrastructure predictors are created and combined with the environmental and socioeconomic predictors from Dataset19.

The resulting infrastructure-enhanced dataset is referred to as Dataset20.

※Related dissertation sections
 - Section 2.6 (Public Geospatial Data as Predictors)
 - Section 3.1 (Research Approach—Data Understanding and Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 1. Load Dataset19
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import numpy as np
import pandas as pd
import geopandas as gpd

base_dir = (
    "/content/drive/MyDrive/Dissertation/Experiments/"
    "최종실험(수정)/Dataset_Build"
)

dataset19_path = os.path.join(
    base_dir,
    "08_Final_Datasets",
    "dataset19_s1_s2_worldcover_population_nightlight.gpkg"
)

dataset19 = gpd.read_file(
    dataset19_path,
    layer="dataset19_environmental_features"
)

print("Dataset19 shape:", dataset19.shape)
print("Unique grid IDs:", dataset19["grid_id"].nunique())
print("CRS:", dataset19.crs)

print("\nColumns:")
print(dataset19.columns.tolist())

display(dataset19.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset19 shape: (4985, 23)
Unique grid IDs: 4985
CRS: EPSG:27700

Columns:
['grid_id', 'lte_min_rsrp', 'lte_mean_rsrp', 'lte_median_rsrp', 'lte_point_count', 'lte_signal_class', 'nr_min_rsrp', 'nr_mean_rsrp', 'nr_median_rsrp', 'nr_point_count', 'nr_signal_class', 'VV', 'VH', 'NDVI', 'NDBI', 'population', 'nightlight', 'tree_ratio', 'grass_ratio', 'crop_ratio', 'builtup_ratio', 'water_ratio', 'geometry']


,grid_id,lte_min_rsrp,lte_mean_rsrp,lte_median_rsrp,lte_point_count,lte_signal_class,nr_min_rsrp,nr_mean_rsrp,nr_median_rsrp,nr_point_count,...,NDVI,NDBI,population,nightlight,tree_ratio,grass_ratio,crop_ratio,builtup_ratio,water_ratio,geometry
0,3975,-55.64,-46.228794,-45.48,937.0,Excellent,-128.64,-89.648030,-92.37,1755.0,...,0.524089,-0.080509,1594.242380,8.347392,0.195237,0.371513,0.235360,0.197890,0.000000,"POLYGON ((426000 576000, 426000 577000, 425000..."
1,3976,-81.71,-68.405396,-69.13,1609.0,Excellent,-130.87,-92.905040,-92.43,2657.0,...,0.463781,-0.047096,3061.673353,14.387132,0.222356,0.254962,0.000000,0.522511,0.000000,"POLYGON ((427000 576000, 427000 577000, 426000..."
2,4072,-89.99,-83.527565,-84.71,1134.0,Good,-138.30,-100.509040,-96.83,2055.0,...,0.527302,-0.078790,2471.479708,7.935724,0.237174,0.341157,0.000000,0.421669,0.000000,"POLYGON ((427000 575000, 427000 576000, 426000..."
3,4166,-74.81,-64.096870,-62.38,624.0,Excellent,-128.37,-90.326965,-93.48,1207.0,...,0.585328,-0.124289,173.731904,5.853407,0.357394,0.422191,0.067523,0.152720,0.000057,"POLYGON ((424000 574000, 424000 575000, 423000..."
4,4167,-74.29,-70.279630,-70.44,216.0,Excellent,-131.27,-91.655655,-94.24,361.0,...,0.561595,-0.088098,40.148841,3.391786,0.124595,0.423217,0.424147,0.027295,0.000746,"POLYGON ((425000 574000, 425000 575000, 424000..."


## 1. Load and Inspect the Input Data

Dataset19, containing the signal labels and 11 environmental and socioeconomic predictors, is loaded as the baseline input.

The raw OpenCellID file is then inspected for its structure, radio-technology distribution, operator identifiers, missing coordinates, and coordinate ranges.

※Related dissertation sections
 - Section 3.1 (Research Approach—Data Understanding)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 2. Inspect OpenCellID Raw Data
# ============================================================

opencellid_path = (
    "/content/drive/MyDrive/Dissertation/"
    "OpencellID/234_raw.csv"
)

# Validate source file
if not os.path.exists(opencellid_path):
    raise FileNotFoundError(
        f"OpenCellID source file not found: {opencellid_path}"
    )

# Read the first five rows
ocid_sample = pd.read_csv(
    opencellid_path,
    nrows=5,
    low_memory=False
)

print("OpenCellID file:", opencellid_path)
print("File exists:", os.path.exists(opencellid_path))
print("Sample shape:", ocid_sample.shape)

print("\nColumns:")
print(ocid_sample.columns.tolist())

print("\nFirst five rows:")
display(ocid_sample)

OpenCellID file: /content/drive/MyDrive/Dissertation/OpencellID/234_raw.csv
File exists: True
Sample shape: (5, 14)

Columns:
['radio', 'mcc', 'net', 'area', 'cell', 'unit', 'lon', 'lat', 'range', 'samples', 'changeable', 'created', 'updated', 'averageSignal']

First five rows:


,radio,mcc,net,area,cell,unit,lon,lat,range,samples,changeable,created,updated,averageSignal
0,GSM,234,10,21574,33452,0,-1.1264,52.3113,1278,23,1,1459814306,1741057142,0
1,GSM,234,15,139,2243,0,-0.4484,51.5013,1043,31,1,1311882509,1751652009,0
2,GSM,234,30,2534,58455,0,-1.3392,51.2564,1028,25,1,1459707931,1741455721,0
3,GSM,234,30,2534,58454,0,-1.3397,51.2650,1000,18,1,1459707931,1741278782,0
4,UMTS,234,20,46,8848743,185,-1.2573,52.3876,1960,50,1,1345445129,1764704651,0


In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 3. Load and Inspect Full OpenCellID Data
# ============================================================

ocid_columns = [
    "radio",
    "mcc",
    "net",
    "area",
    "cell",
    "unit",
    "lon",
    "lat",
    "range",
    "samples",
    "updated"
]

ocid = pd.read_csv(
    opencellid_path,
    usecols=ocid_columns,
    low_memory=False
)

print("OpenCellID shape:", ocid.shape)

print("\nRadio technology distribution:")
print(ocid["radio"].value_counts(dropna=False))

print("\nUnique operators (net):")
print(ocid["net"].nunique(dropna=True))

print("\nMissing coordinates:")
print(ocid[["lon", "lat"]].isna().sum())

print("\nCoordinate ranges:")
print(ocid[["lon", "lat"]].agg(["min", "max"]))

display(ocid.head())

OpenCellID shape: (135807, 11)

Radio technology distribution:
radio
LTE     110030
GSM      13630
UMTS      7959
NR        4188
Name: count, dtype: int64

Unique operators (net):
12

Missing coordinates:
lon    0
lat    0
dtype: int64

Coordinate ranges:
        lon      lat
min -8.1609  49.1753
max  1.7667  60.2885


,radio,mcc,net,area,cell,unit,lon,lat,range,samples,updated
0,GSM,234,10,21574,33452,0,-1.1264,52.3113,1278,23,1741057142
1,GSM,234,15,139,2243,0,-0.4484,51.5013,1043,31,1751652009
2,GSM,234,30,2534,58455,0,-1.3392,51.2564,1028,25,1741455721
3,GSM,234,30,2534,58454,0,-1.3397,51.2650,1000,18,1741278782
4,UMTS,234,20,46,8848743,185,-1.2573,52.3876,1960,50,1764704651


## 2. Spatial Preparation and Integration

OpenCellID longitude and latitude records are converted from WGS84 (EPSG:4326) to the British National Grid (EPSG:27700).

Duplicate cellular records are removed using their network identifiers and coordinates. The remaining records are spatially assigned to the labelled 1 km grid cells using a point-in-polygon join.

※ Related dissertation sections
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 4. Prepare Grid and OpenCellID Geometries
# ============================================================

# Dataset19 already contains valid EPSG:27700 geometry
grid_gdf = dataset19.copy()

# Convert OpenCellID records to point geometries
ocid_gdf = gpd.GeoDataFrame(
    ocid.copy(),
    geometry=gpd.points_from_xy(
        ocid["lon"],
        ocid["lat"]
    ),
    crs="EPSG:4326"
).to_crs(epsg=27700)

# Validate geometries
print("Dataset19 grids:", f"{len(grid_gdf):,}")
print("Unique grid IDs:", grid_gdf["grid_id"].nunique())
print("Grid CRS:", grid_gdf.crs)
print("Invalid grid geometries:", (~grid_gdf.geometry.is_valid).sum())

print("\nOpenCellID records:", f"{len(ocid_gdf):,}")
print("OpenCellID CRS:", ocid_gdf.crs)
print("Missing point geometries:", ocid_gdf.geometry.isna().sum())
print("Invalid point geometries:", (~ocid_gdf.geometry.is_valid).sum())

Dataset19 grids: 4,985
Unique grid IDs: 4985
Grid CRS: EPSG:27700
Invalid grid geometries: 0

OpenCellID records: 135,807
OpenCellID CRS: EPSG:27700
Missing point geometries: 0
Invalid point geometries: 0


In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 5. Clean and Spatially Join OpenCellID Records
# ============================================================

# Remove duplicate cellular records using network identifiers
ocid_clean = ocid_gdf.drop_duplicates(
    subset=[
        "radio", "mcc", "net",
        "area", "cell", "unit",
        "lon", "lat"
    ]
).copy()

print("Records before deduplication:", f"{len(ocid_gdf):,}")
print("Records after deduplication :", f"{len(ocid_clean):,}")
print("Duplicates removed          :", f"{len(ocid_gdf) - len(ocid_clean):,}")

# Spatially join records to labelled 1 km grids
ocid_joined = gpd.sjoin(
    ocid_clean[
        ["radio", "net", "geometry"]
    ],
    grid_gdf[
        ["grid_id", "geometry"]
    ],
    how="inner",
    predicate="within"
)

print("\nOpenCellID records joined:", f"{len(ocid_joined):,}")
print(
    "Grids containing OpenCellID records:",
    f"{ocid_joined['grid_id'].nunique():,}"
)

display(ocid_joined.head())

Records before deduplication: 135,807
Records after deduplication : 135,807
Duplicates removed          : 0

OpenCellID records joined: 35,207
Grids containing OpenCellID records: 3,661


,radio,net,geometry,index_right,grid_id
0,GSM,10,POINT (459654.353 268491.182),3605,71308
1,GSM,15,POINT (507794.586 179181.366),4731,96600
5,GSM,15,POINT (396963.12 303327.866),2818,59705
6,GSM,15,POINT (419204.094 281083.318),3441,67035
7,GSM,15,POINT (419320.097 282718.935),3417,66697


## 3. Construction of Cellular-Infrastructure Features

The spatially matched OpenCellID records are aggregated by `grid_id` to create six infrastructure predictors:

- 'cell_count': total number of OpenCellID records;
- 'operator_count': number of unique network operators;
- 'gsm_count': number of GSM records;
- 'umts_count': number of UMTS records;
- 'lte_count': number of LTE records; and
- 'nr_count': number of 5G NR records.

The technology-specific counts are validated against the total record count for each grid.

※ Related dissertation sections
 - Section 2.6 (Public Geospatial Data as Predictors)
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction)
 - Appendix C (Predictor Definitions).

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 6. Create Grid-Level Infrastructure Features
# ============================================================

# Total OpenCellID records per grid
cell_count = (
    ocid_joined
    .groupby("grid_id")
    .size()
    .rename("cell_count")
)

# Number of unique operators per grid
operator_count = (
    ocid_joined
    .groupby("grid_id")["net"]
    .nunique()
    .rename("operator_count")
)

# Technology-specific record counts
tech_counts = (
    ocid_joined
    .groupby(["grid_id", "radio"])
    .size()
    .unstack(fill_value=0)
)

for tech in ["GSM", "UMTS", "LTE", "NR"]:
    if tech not in tech_counts.columns:
        tech_counts[tech] = 0

tech_counts = tech_counts[
    ["GSM", "UMTS", "LTE", "NR"]
].rename(
    columns={
        "GSM": "gsm_count",
        "UMTS": "umts_count",
        "LTE": "lte_count",
        "NR": "nr_count"
    }
)

# Combine infrastructure features
infrastructure_features = pd.concat(
    [
        cell_count,
        operator_count,
        tech_counts
    ],
    axis=1
).reset_index()

# Validate total count against technology counts
tech_total = infrastructure_features[
    ["gsm_count", "umts_count", "lte_count", "nr_count"]
].sum(axis=1)

if not tech_total.equals(infrastructure_features["cell_count"]):
    raise ValueError("Technology counts do not match total cell counts.")

print("Infrastructure feature shape:", infrastructure_features.shape)
print("Unique grid IDs:", infrastructure_features["grid_id"].nunique())

print("\nColumns:")
print(infrastructure_features.columns.tolist())

display(infrastructure_features.head())

Infrastructure feature shape: (3661, 7)
Unique grid IDs: 3661

Columns:
['grid_id', 'cell_count', 'operator_count', 'gsm_count', 'umts_count', 'lte_count', 'nr_count']


,grid_id,cell_count,operator_count,gsm_count,umts_count,lte_count,nr_count
0,3975,5,2,0,0,5,0
1,4166,4,2,0,0,4,0
2,4168,2,2,0,0,2,0
3,4169,1,1,0,0,1,0
4,4270,5,3,0,0,5,0


## 4. Creation and Export of Dataset20

The six OpenCellID infrastructure predictors are merged with Dataset19 using 'grid_id'.

Grid cells without matched OpenCellID records are assigned zero values. The completed Dataset20 contains the original 11 environmental and socioeconomic predictors together with the six infrastructure predictors.

The final dataset is validated and exported in CSV and GeoPackage formats.

※ Related dissertation sections
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction)
 - Appendix C (Reproducibility and Predictor Definitions).

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 7. Merge Infrastructure Features with Dataset19
# ============================================================

infra_cols = [
    "cell_count",
    "operator_count",
    "gsm_count",
    "umts_count",
    "lte_count",
    "nr_count"
]

# Merge infrastructure features
dataset20 = dataset19.merge(
    infrastructure_features,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Assign zero to grids without OpenCellID records
dataset20[infra_cols] = (
    dataset20[infra_cols]
    .fillna(0)
    .astype("int32")
)

# Validate merge
if len(dataset20) != len(dataset19):
    raise ValueError("The number of rows changed during merging.")

if dataset20["grid_id"].nunique() != len(dataset20):
    raise ValueError("Duplicate grid IDs were found in Dataset20.")

print("Dataset20 shape:", dataset20.shape)
print("Unique grid IDs:", dataset20["grid_id"].nunique())

print("\nMissing infrastructure values:")
print(dataset20[infra_cols].isna().sum())

print("\nInfrastructure summary:")
display(dataset20[infra_cols].describe())

display(dataset20.head())

Dataset20 shape: (4985, 29)
Unique grid IDs: 4985

Missing infrastructure values:
cell_count        0
operator_count    0
gsm_count         0
umts_count        0
lte_count         0
nr_count          0
dtype: int64

Infrastructure summary:


,cell_count,operator_count,gsm_count,umts_count,lte_count,nr_count
count,4985.000000,4985.000000,4985.000000,4985.000000,4985.000000,4985.000000
mean,7.062588,1.839920,0.676630,0.373721,5.791174,0.221063
std,12.290120,1.483068,2.821279,1.667408,9.771133,1.228702
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.000000,2.000000,0.000000,0.000000,3.000000,0.000000
75%,9.000000,3.000000,1.000000,0.000000,8.000000,0.000000
max,261.000000,5.000000,141.000000,62.000000,203.000000,43.000000


,grid_id,lte_min_rsrp,lte_mean_rsrp,lte_median_rsrp,lte_point_count,lte_signal_class,nr_min_rsrp,nr_mean_rsrp,nr_median_rsrp,nr_point_count,...,crop_ratio,builtup_ratio,water_ratio,geometry,cell_count,operator_count,gsm_count,umts_count,lte_count,nr_count
0,3975,-55.64,-46.228794,-45.48,937.0,Excellent,-128.64,-89.648030,-92.37,1755.0,...,0.235360,0.197890,0.000000,"POLYGON ((426000 576000, 426000 577000, 425000...",5,2,0,0,5,0
1,3976,-81.71,-68.405396,-69.13,1609.0,Excellent,-130.87,-92.905040,-92.43,2657.0,...,0.000000,0.522511,0.000000,"POLYGON ((427000 576000, 427000 577000, 426000...",0,0,0,0,0,0
2,4072,-89.99,-83.527565,-84.71,1134.0,Good,-138.30,-100.509040,-96.83,2055.0,...,0.000000,0.421669,0.000000,"POLYGON ((427000 575000, 427000 576000, 426000...",0,0,0,0,0,0
3,4166,-74.81,-64.096870,-62.38,624.0,Excellent,-128.37,-90.326965,-93.48,1207.0,...,0.067523,0.152720,0.000057,"POLYGON ((424000 574000, 424000 575000, 423000...",4,2,0,0,4,0
4,4167,-74.29,-70.279630,-70.44,216.0,Excellent,-131.27,-91.655655,-94.24,361.0,...,0.424147,0.027295,0.000746,"POLYGON ((425000 574000, 425000 575000, 424000...",0,0,0,0,0,0


In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 8. Save Final Dataset20
# ============================================================

dataset20_output_dir = os.path.join(
    base_dir,
    "08_Final_Datasets"
)

csv_path = os.path.join(
    dataset20_output_dir,
    "dataset20_final.csv"
)

gpkg_path = os.path.join(
    dataset20_output_dir,
    "dataset20_final.gpkg"
)

# Save tabular dataset without geometry
dataset20.drop(
    columns="geometry"
).to_csv(
    csv_path,
    index=False
)

# Save spatial dataset with geometry
dataset20.to_file(
    gpkg_path,
    layer="dataset20_final",
    driver="GPKG"
)

print("Final Dataset20 shape:", dataset20.shape)
print("Unique grid IDs:", dataset20["grid_id"].nunique())

print("\nCSV saved:")
print(csv_path)

print("\nGeoPackage saved:")
print(gpkg_path)

print("\nInfrastructure dtypes:")
print(dataset20[infra_cols].dtypes)

Final Dataset20 shape: (4985, 29)
Unique grid IDs: 4985

CSV saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets/dataset20_final.csv

GeoPackage saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets/dataset20_final.gpkg

Infrastructure dtypes:
cell_count        int32
operator_count    int32
gsm_count         int32
umts_count        int32
lte_count         int32
nr_count          int32
dtype: object
